# Entrenamiento final — multilingual E5 base, dataset v6
Versión elegida por el usuario para mostrar las pruebas realizadas. Referencia: checkpoint-126 (época 6), F1 macro de validación 65,57 %, accuracy@1 87,50 %, F1 micro 82,35 %. Evaluación: 16 consultas de 12 expedientes frente a 444 artículos. Las etiquetas siguen siendo débiles; estos resultados no son una certificación jurídica ni un resultado de test.

No se incorporan limpieza v7, limpieza de encabezados ni reordenador: empeoraron las pruebas o no mejoraron el criterio elegido. Se usan los textos originales del v6. Sentence Transformers es la biblioteca; el modelo es **intfloat/multilingual-e5-base**.

Por defecto **REENTRENAR=False**: recupera y evalúa el checkpoint ganador de la sesión. Si ya se perdió, recupera sus archivos o cambia REENTRENAR=True para repetir la receta; repetirla no garantiza métricas idénticas. Test y guardado final de pesos desactivados.


In [ ]:
%pip -q install sentence-transformers==5.1.1 transformers==4.56.2 datasets==4.1.1 accelerate==1.10.1 scikit-learn


In [ ]:
from pathlib import Path
import sys, json, importlib, platform
from google.colab import drive
drive.mount('/content/drive')
CODE=Path('/content/codigo_entrenamiento_final_e5'); CODE.mkdir(exist_ok=True)
(CODE/'dataset_utils.py').write_text('"""Normalización, catálogo y partición compartidos por pipeline y entrenamiento."""\nimport hashlib\nimport html\nimport json\nimport re\nimport unicodedata\nfrom collections import defaultdict\nfrom pathlib import Path\n\nSUFFIX = r\'(?:bis|ter|quater|quinquies|sexies|septies)\'\nNUMBER = rf\'\\d+(?:\\s*{SUFFIX})?\'\n\ndef normalize_text(value):\n    return re.sub(r\'\\s+\', \' \', unicodedata.normalize(\'NFKC\', str(value or \'\'))).strip()\n\ndef article_number(value):\n    text = normalize_text(value).lower()\n    if re.fullmatch(r\'\\d+\\.0\', text): text = text[:-2]\n    match = re.fullmatch(rf\'(\\d+)\\s*({SUFFIX})?\', text)\n    if not match: raise ValueError(f\'Número inválido: {value!r}\')\n    return str(int(match[1])) + (f\' {match[2]}\' if match[2] else \'\')\n\ndef fingerprint(value):\n    return hashlib.sha256(normalize_text(value).casefold().encode(\'utf-8\')).hexdigest()\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1048576), b\'\'): digest.update(block)\n    return digest.hexdigest()\n\ndef read_jsonl(path):\n    with Path(path).open(encoding=\'utf-8\') as stream:\n        return [json.loads(line) for line in stream if line.strip()]\n\ndef write_jsonl(path, rows):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    with temp.open(\'w\', encoding=\'utf-8\') as stream:\n        for row in rows: stream.write(json.dumps(row, ensure_ascii=False) + \'\\n\')\n    temp.replace(path)\n\ndef load_catalog(path):\n    data = json.loads(Path(path).read_text(encoding=\'utf-8-sig\'))\n    if isinstance(data, dict):\n        data = [{\'numero_articulo\': k, \'contenido\': v, \'norma_sigla\': \'CP\'} for k,v in data.items()]\n    if not isinstance(data, list): raise ValueError(\'Catálogo: se espera lista de objetos o mapa CP número→texto.\')\n    catalog = {}\n    for row in data:\n        if normalize_text(row.get(\'norma_sigla\')).upper() != \'CP\': continue\n        number = article_number(row[\'numero_articulo\'])\n        text = normalize_text(html.unescape(re.sub(r\'<[^>]+>\', \' \', str(row.get(\'contenido\') or \'\'))))\n        if not text: raise ValueError(f\'CP {number}: contenido vacío\')\n        if number in catalog and catalog[number] != text: raise ValueError(f\'CP {number}: textos incompatibles\')\n        catalog[number] = text\n    if not catalog: raise ValueError(\'No hay artículos con norma_sigla=CP\')\n    return catalog\n\ndef grouped_split(rows, seed=42):\n    """Unión transitiva por resolución, documento y texto; se divide antes de aumentar."""\n    parent = list(range(len(rows)))\n    def find(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]; i = parent[i]\n        return i\n    seen = {}\n    for i,row in enumerate(rows):\n        source = row.get(\'fuente_id\')\n        if source is None or normalize_text(source) == \'\': raise ValueError(\'Falta fuente_id\')\n        keys = [(\'source\', str(source)), (\'text\', fingerprint(row[\'hechos\']))]\n        if row.get(\'documento_hash\'): keys.append((\'document\', row[\'documento_hash\']))\n        case = normalize_text(row.get(\'nro_expediente\')).casefold()\n        if case and re.search(r\'\\d{4}\',case): keys.append((\'case\',case))\n        for key in keys:\n            if key in seen: parent[find(i)] = find(seen[key])\n            else: seen[key] = i\n    groups = defaultdict(list)\n    for i in range(len(rows)): groups[find(i)].append(i)\n    result = []\n    for indices in groups.values():\n        # Contenido estable: independencia del orden de archivos/filas.\n        identity = sorted({str(rows[i][\'fuente_id\']) for i in indices})\n        group = fingerprint(json.dumps(identity, ensure_ascii=False))\n        value = int(fingerprint(f\'{seed}:{group}\')[:16],16) / 2**64\n        split = \'train\' if value < .8 else (\'validation\' if value < .9 else \'test\')\n        for i in indices: result.append({**rows[i], \'grupo_id\': group, \'split\': split})\n    return sorted(result, key=lambda r:(str(r[\'fuente_id\']),r[\'articulo\'],r[\'hechos\']))\n',encoding='utf-8')
(CODE/'group_metrics.py').write_text('"""Métricas por consulta y con igual peso por expediente; sin consultar test."""\nfrom collections import defaultdict\nfrom dataset_utils import fingerprint\n\ndef group_metrics(rows, records):\n    import numpy as np\n    from sklearn.preprocessing import MultiLabelBinarizer\n    from sklearn.metrics import f1_score\n    memberships = defaultdict(set)\n    for row in rows:\n        memberships[fingerprint(row[\'hechos\'])].add(str(row[\'grupo_id\']))\n    if any(len(v) != 1 for v in memberships.values()):\n        raise ValueError(\'Consulta asociada a varios grupos: revisar agrupación\')\n    counts = defaultdict(int)\n    for record in records:\n        counts[next(iter(memberships[record[\'query_id\']]))] += 1\n    weights = [1 / counts[next(iter(memberships[r[\'query_id\']]))] for r in records]\n    truth = [set(r[\'positivos_conocidos\']) for r in records]\n    pred = [{r[\'top10\'][0]} for r in records]\n    labels = sorted(set().union(*truth, *pred))\n    encoder = MultiLabelBinarizer(classes=labels)\n    y = encoder.fit_transform(truth); p = encoder.transform(pred)\n    result = {\'expedientes\': len(counts), \'consultas\': len(records),\n              \'accuracy_top1_por_expediente\': float(np.average([r[\'acierto_top1\'] for r in records], weights=weights))}\n    for avg in (\'macro\', \'micro\', \'weighted\'):\n        result[\'f1_\' + avg + \'_por_expediente\'] = float(f1_score(y, p, average=avg, sample_weight=weights, zero_division=0))\n    return result\n',encoding='utf-8')
(CODE/'train_e5_base.py').write_text('"""Entrenamiento reproducible con checkpoints completos; compatible con notebook Colab."""\nimport argparse\nimport json\nimport os\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom dataset_utils import file_hash,fingerprint,read_jsonl\n\nBASE_MODEL=\'intfloat/multilingual-e5-base\'\n\ndef query_text(text):\n    return \'query: \'+text\n\ndef passage_text(text):\n    return \'passage: \'+text\nos.environ.setdefault(\'USE_TF\',\'0\')\n\ndef validate_data(rows,catalog):\n    grouped=defaultdict(list); seen_text={}; seen_source={}; seen_group={}\n    positives=defaultdict(set)\n    for row in rows:\n        split=row[\'split\'];number=row[\'articulo\'];text=row[\'hechos\']\n        if split not in (\'train\',\'validation\',\'test\'): raise ValueError(f\'Split inválido: {split}\')\n        if not text or number not in catalog: raise ValueError(\'Hecho vacío o artículo sin catálogo\')\n        if split!=\'train\' and row.get(\'tipo\')!=\'original\': raise ValueError(\'Aumento fuera de train\')\n        for value,seen in ((fingerprint(text),seen_text),(str(row[\'fuente_id\']),seen_source),(row[\'grupo_id\'],seen_group)):\n            if value in seen and seen[value]!=split: raise ValueError(\'Fuga entre splits\')\n            seen[value]=split\n        grouped[split].append(row); positives[fingerprint(text)].add(number)\n    # MNRL de un positivo: no enfrentar anclas con varios artículos conocidos.\n    ambiguous={key for key,values in positives.items() if len(values)>1}\n    # Conservar pares multietiqueta: la pérdida enmascara TODOS sus positivos conocidos.\n    for split in (\'train\',\'validation\',\'test\'):\n        if not grouped[split]: raise ValueError(f\'Split {split} vacío; ampliar datos/revisar grupos\')\n    if len({catalog[r[\'articulo\']] for r in grouped[\'train\']})<2: raise ValueError(\'Se necesitan dos positivos diferentes en train\')\n    print(f\'Anclas multietiqueta conservadas: {len(ambiguous)}; pares train: {len(grouped["train"])}\')\n    return grouped\n\ndef ranking(model,rows,catalog,batch_size=32,seen_articles=None,prediction_path=None):\n    import numpy as np\n    numbers=sorted(catalog);corpus=model.encode([passage_text(catalog[n]) for n in numbers],normalize_embeddings=True,batch_size=batch_size,show_progress_bar=True)\n    queries={};positive=defaultdict(set)\n    for row in rows:\n        key=fingerprint(row[\'hechos\']);queries[key]=row[\'hechos\'];positive[key].add(row[\'articulo\'])\n    keys=list(queries);scores=model.encode([query_text(queries[k]) for k in keys],normalize_embeddings=True,batch_size=batch_size,show_progress_bar=True)\n    totals=defaultdict(float);by_article=defaultdict(list); true_sets=[]; predicted_sets=[];records=[];cohorts=defaultdict(list)\n    for start in range(0,len(keys),batch_size):\n        orders=np.argsort(-(scores[start:start+batch_size]@corpus.T),axis=1)\n        for offset,order in enumerate(orders):\n            relevant=positive[keys[start+offset]];ranked=[numbers[i] for i in order]\n            first=min(ranked.index(n)+1 for n in relevant);totals[\'mrr\']+=1/first\n            true_sets.append(relevant);predicted_sets.append({ranked[0]});totals[\'accuracy@1\']+=float(ranked[0] in relevant)\n            dcg=sum(1/np.log2(rank+2) for rank,n in enumerate(ranked[:10]) if n in relevant)\n            ideal=sum(1/np.log2(rank+2) for rank in range(min(10,len(relevant))))\n            totals[\'ndcg@10\']+=dcg/ideal\n            record={\'query_id\':keys[start+offset],\'hechos\':queries[keys[start+offset]],\'positivos_conocidos\':sorted(relevant),\n                    \'top10\':ranked[:10],\'acierto_top1\':ranked[0] in relevant,\'primer_positivo_rank\':first}\n            if seen_articles is not None:\n                cohort=\'con_positivos_no_vistos\' if relevant-set(seen_articles) else \'solo_positivos_vistos\'\n                cohorts[cohort].append(float(ranked[0] in relevant));record[\'cohorte\']=cohort\n            records.append(record)\n            for k in (1,3,5):\n                retrieved=set(ranked[:k]);hits=len(retrieved&relevant)\n                totals[f\'recall@{k}\']+=hits/len(relevant);totals[f\'precision@{k}\']+=hits/min(k,len(numbers))\n            for number in relevant: by_article[number].append(float(number in ranked[:3]))\n    result={key:value/len(keys) for key,value in totals.items()}\n    result[\'macro_article_recall@3\']=float(np.mean([np.mean(v) for v in by_article.values()]))\n    from sklearn.preprocessing import MultiLabelBinarizer\n    from sklearn.metrics import f1_score, precision_score, recall_score\n    labels=sorted(set().union(*true_sets,*predicted_sets))\n    mlb=MultiLabelBinarizer(classes=labels)\n    truth=mlb.fit_transform(true_sets);predictions=mlb.transform(predicted_sets)\n    for avg in (\'micro\',\'macro\',\'weighted\'):\n        result[f\'f1_{avg}_top1\']=float(f1_score(truth,predictions,average=avg,zero_division=0))\n        result[f\'precision_{avg}_top1\']=float(precision_score(truth,predictions,average=avg,zero_division=0))\n        result[f\'recall_{avg}_top1\']=float(recall_score(truth,predictions,average=avg,zero_division=0))\n    from group_metrics import group_metrics\n    result[\'por_expediente\']=group_metrics(rows,records)\n    result[\'queries\']=len(keys);result[\'candidates\']=len(numbers)\n    result[\'por_articulo_recall@3\']={number:{\'soporte_queries\':len(values),\'recall@3\':float(np.mean(values))} for number,values in by_article.items()}\n    if cohorts:result[\'cohortes\']={name:{\'queries\':len(values),\'accuracy@1\':float(np.mean(values))} for name,values in cohorts.items()}\n    if prediction_path:\n        from dataset_utils import write_jsonl\n        write_jsonl(prediction_path,records)\n    return result\n\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass MultiPositiveRankingLoss(nn.Module):\n    """Contraste query→artículo: distribuir probabilidad entre positivos conocidos.\n\n    label[0] es el ID del artículo de esta fila; label[1:] marca todos los\n    artículos válidos para su ancla. No se consideran negativos otros positivos.\n    """\n    def __init__(self,model,scale=20.0,number_count=None,article_weights=None):\n        super().__init__();self.model=model;self.scale=scale;self.number_count=number_count\n        self.register_buffer(\'article_weights\',torch.tensor(article_weights,dtype=torch.float32) if article_weights is not None else None)\n    @staticmethod\n    def objective(scores,labels,number_count=None,article_weights=None):\n        count=number_count or labels.shape[1]-1\n        document_ids=[labels[:,0].long()]\n        for column in range(1+count,labels.shape[1]):document_ids.append(labels[:,column].long())\n        document_ids=torch.cat(document_ids)\n        positive_mask=labels[:,1:1+count].bool()[:,document_ids]\n        if not positive_mask.diagonal().all():raise ValueError(\'Falta positivo diagonal\')\n        log_probs=F.log_softmax(scores.float(),dim=1)\n        # Cada positivo conocido recibe peso; no basta optimizar el más fácil.\n        per_anchor=-(log_probs.masked_fill(~positive_mask,0).sum(1)/positive_mask.sum(1))\n        if article_weights is not None:\n            positives=labels[:,1:1+count].float()\n            weights=(positives@article_weights.to(scores.device))/positives.sum(1)\n            return (per_anchor*weights).sum()/weights.sum()\n        return per_anchor.mean()\n    def forward(self,sentence_features,labels):\n        anchors=F.normalize(self.model(sentence_features[0])[\'sentence_embedding\'],dim=1)\n        documents=F.normalize(self.model(sentence_features[1])[\'sentence_embedding\'],dim=1)\n        if len(sentence_features)>2:\n            documents=torch.cat([documents]+[F.normalize(self.model(features)[\'sentence_embedding\'],dim=1) for features in sentence_features[2:]],dim=0)\n        return self.objective(anchors@documents.T*self.scale,labels,self.number_count,self.article_weights)\n    def get_config_dict(self):return {\'scale\':self.scale,\'positives\':\'all-known-labels\',\'number_count\':self.number_count,\'class_balanced\':self.article_weights is not None}\n\n\ndef train(run_dir,output_dir,epochs=12,batch_size=16,max_seq_length=512,seed=42,initial_model=None,approved_only=False,learning_rate=2e-5,scale=20.0,patience=3,evaluate_test=False,negative_reviews=None,selection_metric=\'f1_macro\',class_balance_loss=False):\n    import torch\n    from datasets import Dataset\n    from sentence_transformers import SentenceTransformer,SentenceTransformerTrainer,SentenceTransformerTrainingArguments,losses\n    from sentence_transformers.training_args import BatchSamplers\n    from sentence_transformers.evaluation import InformationRetrievalEvaluator,SentenceEvaluator\n    from transformers import set_seed,EarlyStoppingCallback\n    from transformers.trainer_utils import get_last_checkpoint\n    if learning_rate<=0 or scale<=0 or patience<1 or selection_metric not in (\'f1_macro\',\'f1_macro_group\',\'ndcg\',\'mrr\'): raise ValueError(\'Hiperparámetros inválidos\')\n    directory=Path(run_dir);output=Path(output_dir)\n    if \'/drive/\' in output.as_posix().casefold(): raise ValueError(\'Checkpoints deben ser locales; guardado en Drive desactivado\')\n    output.mkdir(parents=True,exist_ok=True)\n    rows=read_jsonl(directory/\'pares_aumentados.jsonl\');catalog=json.loads((directory/\'catalogo_cp.json\').read_text(encoding=\'utf-8\'))\n    if approved_only: rows=[r for r in rows if r.get(\'estado_revision\')==\'aprobado\']\n    data=validate_data(rows,catalog);set_seed(seed)\n    config={\'dataset_sha256\':file_hash(directory/\'pares_aumentados.jsonl\'),\'catalog_sha256\':file_hash(directory/\'catalogo_cp.json\'),\n        \'recipe\':\'multipositive-e5-base-v1\',\'prefixes\':{\'query\':\'query: \',\'passage\':\'passage: \'},\'learning_rate\':learning_rate,\'scale\':scale,\'patience\':patience,\'base_model\':BASE_MODEL,\'initial_model\':initial_model,\'epochs\':epochs,\'batch_size\':batch_size,\'max_seq_length\':max_seq_length,\'seed\':seed,\'approved_only\':approved_only,\n        \'negative_reviews_sha256\':file_hash(negative_reviews) if negative_reviews else None,\'selection_metric\':selection_metric,\'class_balance_loss\':class_balance_loss}\n    config_path=output/\'run_config.json\'\n    if config_path.exists() and json.loads(config_path.read_text(encoding=\'utf-8\'))!=config:\n        raise ValueError(\'Configuración/datos diferentes: usa una carpeta de modelo nueva\')\n    config_path.write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    checkpoint=get_last_checkpoint(str(output))\n    if checkpoint:\n        required=[\'trainer_state.json\',\'optimizer.pt\',\'scheduler.pt\']\n        missing=[name for name in required if not (Path(checkpoint)/name).exists()]\n        if missing: raise ValueError(f\'Checkpoint incompleto: {missing}\')\n    model=SentenceTransformer(checkpoint or initial_model or BASE_MODEL)\n    architecture=model[0].auto_model.config\n    capacity=int(architecture.max_position_embeddings)\n    if architecture.model_type in (\'roberta\',\'xlm-roberta\'):\n        capacity-=int(architecture.pad_token_id or 0)+1\n    print(\'Capacidad de arquitectura:\',capacity,\'Límite predeterminado del tokenizer:\',model.tokenizer.model_max_length)\n    if max_seq_length>capacity: raise ValueError(f\'max_seq_length supera capacidad {capacity}\')\n    model.max_seq_length=max_seq_length\n    import numpy as np\n    token_stats={}\n    for name,texts in [(\'hechos\',list({query_text(r[\'hechos\']) for r in rows})),(\'articulos\',list({passage_text(t) for t in catalog.values()}))]:\n        lengths=np.array([len(model.tokenizer(t,truncation=False)[\'input_ids\']) for t in texts])\n        token_stats[name]={\'p50\':float(np.percentile(lengths,50)),\'p95\':float(np.percentile(lengths,95)),\n            \'max\':int(lengths.max()),\'truncated_fraction\':float((lengths>max_seq_length).mean())}\n    (output/\'tokens.json\').write_text(json.dumps(token_stats,indent=2),encoding=\'utf-8\');print(token_stats)\n    queries={};relevant=defaultdict(set)\n    for row in data[\'validation\']:\n        key=fingerprint(row[\'hechos\']);queries[key]=row[\'hechos\'];relevant[key].add(row[\'articulo\'])\n    evaluator=InformationRetrievalEvaluator({k:query_text(v) for k,v in queries.items()},{k:passage_text(v) for k,v in catalog.items()},dict(relevant),name=\'validation\',\n        accuracy_at_k=[1,3,5],precision_recall_at_k=[1,3,5],mrr_at_k=[10],ndcg_at_k=[10],map_at_k=[10],batch_size=batch_size)\n    if selection_metric!=\'ndcg\':\n        parent_evaluator=evaluator\n        class ValidationMetricsEvaluator(SentenceEvaluator):\n            def __init__(self):\n                super().__init__();self.primary_metric=\'validation_\'+selection_metric;self.greater_is_better=True\n            def __call__(self,model,output_path=None,epoch=-1,steps=-1):\n                result=parent_evaluator(model,output_path=output_path,epoch=epoch,steps=steps)\n                validation_metrics=ranking(model,data[\'validation\'],catalog,batch_size)\n                result[self.primary_metric]=(validation_metrics[\'por_expediente\'][\'f1_macro_por_expediente\'] if selection_metric==\'f1_macro_group\' else validation_metrics[\'f1_macro_top1\' if selection_metric==\'f1_macro\' else \'mrr\'])\n                return result\n        evaluator=ValidationMetricsEvaluator()\n    numbers=sorted(catalog);number_ids={number:i for i,number in enumerate(numbers)}\n    known_positives=defaultdict(set)\n    same_text=defaultdict(set)\n    for number,text in catalog.items():same_text[fingerprint(text)].add(number)\n    for row in data[\'train\']:\n        known_positives[fingerprint(row[\'hechos\'])].update(same_text[fingerprint(catalog[row[\'articulo\']])])\n    labels=[]\n    for row in data[\'train\']:\n        positives=known_positives[fingerprint(row[\'hechos\'])]\n        labels.append([number_ids[row[\'articulo\']]]+[int(n in positives) for n in numbers])\n    columns={\'anchor\':[query_text(r[\'hechos\']) for r in data[\'train\']],\n        \'positive\':[passage_text(catalog[r[\'articulo\']]) for r in data[\'train\']]}\n    if negative_reviews:\n        reviewed={}\n        for row in read_jsonl(negative_reviews):\n            if row.get(\'estado_revision\')!=\'aprobado\' or not row.get(\'revisor\') or not row.get(\'justificacion\'):\n                raise ValueError(\'Negativos sin revisión explícita\')\n            if row[\'query_id\'] in reviewed:raise ValueError(\'Revisión de negativos duplicada\')\n            negatives=row[\'negativos_aprobados\']\n            if not isinstance(negatives,list) or not negatives or len(set(negatives))!=len(negatives):raise ValueError(\'Lista de negativos inválida\')\n            if any(n not in catalog for n in negatives):raise ValueError(\'Negativo sin catálogo\')\n            reviewed[row[\'query_id\']]=negatives\n        sizes={len(reviewed.get(fingerprint(r[\'hechos\']),[])) for r in data[\'train\']}\n        if len(sizes)!=1 or 0 in sizes:raise ValueError(\'Revisar mismo número de negativos para TODAS las anclas train\')\n        for index,row in enumerate(data[\'train\']):\n            negatives=reviewed[fingerprint(row[\'hechos\'])]\n            if set(negatives)&known_positives[fingerprint(row[\'hechos\'])]:raise ValueError(\'Negativo es positivo conocido\')\n            labels[index].extend(number_ids[n] for n in negatives)\n        for offset in range(next(iter(sizes))):\n            columns[f\'negative_{offset+1}\']=[passage_text(catalog[reviewed[fingerprint(r[\'hechos\'])][offset]]) for r in data[\'train\']]\n    columns[\'label\']=labels\n    train_dataset=Dataset.from_dict(columns)\n    article_weights=None\n    if class_balance_loss:\n        from collections import Counter\n        frequency=Counter(n for positives in known_positives.values() for n in positives)\n        maximum=max(frequency.values())\n        article_weights=[min(4.0,(maximum/max(1,frequency[n]))**.5) for n in numbers]\n    # Establecer el nombre exacto de la métrica con el evaluador instalado.\n    if not (output/\'baseline_validation.json\').exists():\n        base=SentenceTransformer(BASE_MODEL);base.max_seq_length=max_seq_length\n        baseline=ranking(base,data[\'validation\'],catalog,batch_size)\n        (output/\'baseline_validation.json\').write_text(json.dumps(baseline,indent=2),encoding=\'utf-8\')\n        del base\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\n    evaluator(model,output_path=str(output));metric=evaluator.primary_metric\n    args=SentenceTransformerTrainingArguments(output_dir=str(output),num_train_epochs=epochs,\n        per_device_train_batch_size=batch_size,per_device_eval_batch_size=batch_size,\n        learning_rate=learning_rate,weight_decay=.01,warmup_ratio=.1,fp16=torch.cuda.is_available(),bf16=False,\n        eval_strategy=\'epoch\',save_strategy=\'epoch\',save_total_limit=1,\n        save_only_model=False,load_best_model_at_end=True,metric_for_best_model=metric,greater_is_better=True,\n        batch_sampler=BatchSamplers.NO_DUPLICATES,seed=seed,data_seed=seed,report_to=\'none\',logging_steps=5,gradient_checkpointing=True)\n    trainer=SentenceTransformerTrainer(model=model,args=args,train_dataset=train_dataset,\n        loss=MultiPositiveRankingLoss(model,scale=scale,number_count=len(numbers),article_weights=article_weights),evaluator=evaluator,\n        callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)])\n    trainer.train(resume_from_checkpoint=checkpoint)\n    # trainer.save_model(str(output/\'final\'))  # Guardado final desactivado hasta evaluar.\n    seen_articles={r[\'articulo\'] for r in data[\'train\']}\n    metrics={\'validation\':ranking(model,data[\'validation\'],catalog,batch_size,seen_articles,output/\'predicciones_validation.jsonl\'),\n        \'supervision\':\'aprobada\' if approved_only else \'débil: candidatos extraídos de citas\'}\n    if (directory/\'validation_referencia_v6.jsonl\').exists():\n        metrics[\'validation_referencia_v6\']=ranking(model,read_jsonl(directory/\'validation_referencia_v6.jsonl\'),catalog,batch_size,seen_articles,output/\'predicciones_validation_referencia.jsonl\')\n    if evaluate_test: metrics[\'test\']=ranking(model,data[\'test\'],catalog,batch_size,seen_articles,output/\'predicciones_test.jsonl\')\n    metrics[\'best_checkpoint\']=trainer.state.best_model_checkpoint\n    metrics[\'epochs_completed\']=trainer.state.epoch\n    metrics[\'train_pairs\']=len(data[\'train\'])\n    (output/\'metrics.json\').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    print(json.dumps(metrics,ensure_ascii=False,indent=2));return model,metrics\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--run-dir\',required=True);p.add_argument(\'--output-dir\',required=True)\n    p.add_argument(\'--epochs\',type=int,default=12);p.add_argument(\'--batch-size\',type=int,default=16);p.add_argument(\'--max-seq-length\',type=int,default=512)\n    p.add_argument(\'--initial-model\');p.add_argument(\'--approved-only\',action=\'store_true\')\n    p.add_argument(\'--learning-rate\',type=float,default=2e-5);p.add_argument(\'--scale\',type=float,default=20);p.add_argument(\'--patience\',type=int,default=3);p.add_argument(\'--seed\',type=int,default=42);p.add_argument(\'--evaluate-test\',action=\'store_true\')\n    p.add_argument(\'--negative-reviews\')\n    p.add_argument(\'--selection-metric\',choices=[\'f1_macro\',\'f1_macro_group\',\'ndcg\',\'mrr\'],default=\'f1_macro\')\n    p.add_argument(\'--class-balance-loss\',action=\'store_true\')\n    a=p.parse_args();train(a.run_dir,a.output_dir,a.epochs,a.batch_size,a.max_seq_length,initial_model=a.initial_model,approved_only=a.approved_only,learning_rate=a.learning_rate,scale=a.scale,patience=a.patience,seed=a.seed,evaluate_test=a.evaluate_test,negative_reviews=a.negative_reviews,selection_metric=a.selection_metric,class_balance_loss=a.class_balance_loss)\n',encoding='utf-8')
sys.path.insert(0,str(CODE))
for name in ('dataset_utils','group_metrics','train_e5_base'):
    sys.modules.pop(name,None)
importlib.invalidate_caches()
import train_e5_base as entrenamiento
from dataset_utils import read_jsonl,file_hash
assert Path(entrenamiento.__file__).resolve()==(CODE/'train_e5_base.py').resolve()
print('Biblioteca: sentence-transformers; modelo:',entrenamiento.BASE_MODEL)
print('Código:',entrenamiento.__file__)


In [ ]:
from collections import Counter
RUN_DIR=Path('/content/drive/MyDrive/datasetFinalDerecho')
CHECKPOINT=Path('/content/tsj_e5_base_v6_mejoras/lr_2e-05_seed_42/checkpoint-126')
OUTPUT=Path('/content/entrenamiento_final_e5_base_v6/lr_2e-5_batch16_seed42')
REPORTS=RUN_DIR/'reportes_validation'/'entrenamiento_final_e5_base'
REENTRENAR=False
EVALUAR_TEST=False
manifest=json.loads((RUN_DIR/'manifest.json').read_text(encoding='utf-8'))
assert file_hash(RUN_DIR/'pares_aumentados.jsonl')==manifest['dataset_sha256']=='b3405bfd8b7ba6bdd9b5235a2c03fdd4f69a4a786330a4e7b3f1f8f39cb85067'
assert file_hash(RUN_DIR/'catalogo_cp.json')==manifest['catalog_sha256']=='38d315a66885497b5b0061caa405b5ba35a048ae531f20d19f155b1e30e5d0f3'
rows=read_jsonl(RUN_DIR/'pares_aumentados.jsonl')
catalog=json.loads((RUN_DIR/'catalogo_cp.json').read_text(encoding='utf-8'))
assert Counter(r['split'] for r in rows)=={'train':327,'validation':18,'test':23}
assert len(catalog)==444
assert all(r['articulo_texto']==catalog[r['articulo']] for r in rows)
data=entrenamiento.validate_data(rows,catalog)
print('Train/validation/test:',{k:len(v) for k,v in data.items()})


## Cargar el ganador o repetir la receta
Receta: batch 16, LR 2e-5, longitud 512, seed 42, pérdida contrastiva con múltiples positivos conocidos, batches sin duplicados, escala 20, warmup 10 %, weight decay 0.01, hasta 12 épocas y paciencia 3. Selección por F1 macro top1 sobre la validación original. Los checkpoints locales contienen modelo, optimizador y scheduler; reanudación requiere conservar el checkpoint completo y la configuración.


In [ ]:
from sentence_transformers import SentenceTransformer
if REENTRENAR:
    modelo,metrics_train=entrenamiento.train(RUN_DIR,OUTPUT,epochs=12,batch_size=16,
        max_seq_length=512,seed=42,learning_rate=2e-5,scale=20,patience=3,
        selection_metric='f1_macro',evaluate_test=False)
    checkpoint_usado=metrics_train['best_checkpoint']
else:
    if not (CHECKPOINT/'modules.json').is_file():
        raise FileNotFoundError('Checkpoint ganador ausente. Ajusta CHECKPOINT a sus archivos recuperados, o activa REENTRENAR=True para repetir la receta. No se sustituye por E5 base sin entrenar.')
    modelo=SentenceTransformer(str(CHECKPOINT),local_files_only=True)
    modelo.max_seq_length=512
    checkpoint_usado=str(CHECKPOINT)
print('Checkpoint usado:',checkpoint_usado)


In [ ]:
import importlib.metadata as metadata
REPORTS.mkdir(parents=True,exist_ok=True)
resultados={'modelo':entrenamiento.BASE_MODEL,'checkpoint':checkpoint_usado,
    'supervision':'débil; etiquetas por citas, pendiente validación jurídica',
    'dataset_sha256':file_hash(RUN_DIR/'pares_aumentados.jsonl'),
    'catalog_sha256':file_hash(RUN_DIR/'catalogo_cp.json'),
    'validation':entrenamiento.ranking(modelo,data['validation'],catalog,16,
        seen_articles={r['articulo'] for r in data['train']},prediction_path=REPORTS/'predicciones_validation.jsonl')}
if EVALUAR_TEST:
    resultados['test']=entrenamiento.ranking(modelo,data['test'],catalog,16,
        seen_articles={r['articulo'] for r in data['train']},prediction_path=REPORTS/'predicciones_test.jsonl')
else:
    print('Test no consultado.')
resultados['entorno']={'python':platform.python_version(),**{n:metadata.version(n) for n in ['sentence-transformers','transformers','datasets','accelerate','torch','scikit-learn']}}
print(json.dumps(resultados,ensure_ascii=False,indent=2))
(REPORTS/'metricas_ejecutadas.json').write_text(json.dumps(resultados,ensure_ascii=False,indent=2),encoding='utf-8')
print('Guardado de pesos finales DESACTIVADO. Los reportes sí se guardaron:',REPORTS)
# modelo.save('/content/drive/MyDrive/modelo_final_E5_base')


## Evidencia histórica reportada
Estos números proceden de las salidas de Colab compartidas por el usuario, no de una ejecución oculta de este notebook.

| Prueba | F1 macro, validación original | Accuracy@1 | Recall@3 |
|---|---:|---:|---:|
| E5 base ganador, checkpoint-126 | 0.655678 | 0.875 | 0.875 |
| Selección por expediente, checkpoint-84 | 0.619048 | 0.875 | 0.9375 |
| Limpieza inicial de encabezados | 0.511111 | 0.8125 | 0.875 |

El 0.727273 de F1 macro de sensibilidad utiliza 14 consultas tras apartar dos; no es comparable como mejora del modelo frente a las 16 originales. La reordenación sin adaptación jurídica empeoró fuertemente; se dejó fuera. El test de esta selección no se ha ejecutado. No se alcanzó un F1 macro comparable de 75–80 %.
